# Facilitator: live hardware demo

Runs the Lab 2 Bell circuit on a real IBM quantum computer.

**Before the workshop**
- Add your IBM Quantum API key as a Colab secret named `IBM_QUANTUM_TOKEN`
  (or set the `IBM_QUANTUM_TOKEN` environment variable when running locally).
- Run this notebook the day before and write down the job ID as a backup.
- Rehearse with `USE_FAKE = True`, which uses a local noisy simulator and
  needs no account.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ms-cc-org/qc-workshop/blob/main/facilitator/hardware_demo.ipynb)

In [ ]:
# Install pinned versions (takes about a minute in Colab; already done locally)
%pip install -q qiskit==2.6.1 qiskit-aer==0.18.2 qiskit-ibm-runtime==0.51.0 matplotlib pylatexenc

In [ ]:
USE_FAKE = False        # True = rehearse offline with a simulated device
BACKUP_JOB_ID = ""      # paste the job ID from your day-before run

In [ ]:
import os, warnings
from qiskit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.visualization import plot_histogram
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

# SamplerV2 is deprecated in qiskit-ibm-runtime 0.50+ but still works with the pinned version
warnings.filterwarnings("ignore", category=DeprecationWarning)

def get_token():
    try:
        from google.colab import userdata
        return userdata.get("IBM_QUANTUM_TOKEN")
    except ImportError:
        return os.environ["IBM_QUANTUM_TOKEN"]

if USE_FAKE:
    from qiskit_ibm_runtime.fake_provider import FakeManilaV2
    backend = FakeManilaV2()
else:
    service = QiskitRuntimeService(channel="ibm_quantum_platform", token=get_token())
    backend = service.least_busy(operational=True, simulator=False)
print("Backend:", backend.name, "-", backend.num_qubits, "qubits")

In [ ]:
bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
bell.measure_all()

# Rewrite the circuit into the device's native gates and physical qubits
isa = generate_preset_pass_manager(backend=backend, optimization_level=1).run(bell)
print("Gates after transpiling:", dict(isa.count_ops()))

In [ ]:
job = Sampler(mode=backend).run([isa], shots=1000)
print("Job ID:", job.job_id())

The next cell waits for the job. If the queue is long, interrupt it and
show the backup job instead.

In [ ]:
counts = job.result()[0].data.meas.get_counts()
print(counts)
plot_histogram(counts, title=f"Bell state on {backend.name}")

In [ ]:
# Backup: results from the day-before run
if BACKUP_JOB_ID and not USE_FAKE:
    old = service.job(BACKUP_JOB_ID)
    backup_counts = old.result()[0].data.meas.get_counts()
    print(old.backend().name, backup_counts)
    display(plot_histogram(backup_counts, title="Backup run"))